<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/future-internet/01_openflow_match_action.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Tablas de flujo OpenFlow: match-action en Python

**Internet del Futuro: Redes Programables** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Entender el modelo *match-action* de una tabla de flujos OpenFlow.
- Simular prioridades, comodines, prefijos IP y el *table-miss*.
- Programar un controlador que aprende direcciones MAC.
- Escribir reglas de firewall y verificarlas contra una traza de tráfico.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

## 1. La tabla de flujos
En SDN el conmutador no decide: consulta una tabla de reglas `(prioridad, match, acciones, contadores)` que el **controlador** instala. Si ninguna regla coincide ocurre un *table-miss* y el paquete sube al controlador (**Packet-In**).

In [ ]:
import ipaddress
from dataclasses import dataclass, field

CAMPOS = ["in_port", "eth_src", "eth_dst", "ip_src", "ip_dst", "ip_proto", "tp_dst"]

def coincide(regla, paquete):
    """Una regla coincide si todos sus campos (los ausentes son comodín) coinciden con el paquete."""
    for k, v in regla.items():
        p = paquete.get(k)
        if k in ("ip_src", "ip_dst") and p is not None:
            if ipaddress.ip_address(p) not in ipaddress.ip_network(v):
                return False
        elif p != v:
            return False
    return True

@dataclass
class Entrada:
    prioridad: int
    match: dict
    acciones: list          # p. ej. ["output:2"], ["drop"], ["controller"]
    paquetes: int = 0

@dataclass
class TablaFlujos:
    entradas: list = field(default_factory=list)
    def agregar(self, prioridad, match, acciones):
        self.entradas.append(Entrada(prioridad, match, acciones))
        self.entradas.sort(key=lambda e: -e.prioridad)
    def procesar(self, paquete):
        for e in self.entradas:
            if coincide(e.match, paquete):
                e.paquetes += 1
                return e.acciones
        return ["controller"]   # table-miss: se envía al controlador (Packet-In)
    def mostrar(self):
        for e in self.entradas:
            print(f"prio={e.prioridad:<5} match={e.match}  ->  {e.acciones}  [{e.paquetes} pkts]")

In [ ]:
t = TablaFlujos()
t.agregar(100, {"ip_proto": 6, "tp_dst": 22}, ["drop"])          # bloquear SSH
t.agregar(50,  {"ip_dst": "10.0.0.0/24"}, ["output:2"])           # LAN interna
t.agregar(0,   {}, ["output:1"])                                  # regla por defecto
print(t.procesar({"ip_dst": "10.0.0.7", "ip_proto": 6, "tp_dst": 22}))
print(t.procesar({"ip_dst": "10.0.0.7", "ip_proto": 6, "tp_dst": 80}))
print(t.procesar({"ip_dst": "8.8.8.8", "ip_proto": 17}))
t.mostrar()

**Pregunta:** ¿qué pasaría si la regla de SSH tuviera prioridad 10 en lugar de 100?

## 2. Una traza de tráfico
Cuatro hosts conectados a los puertos 1–4 del conmutador generan 200 paquetes aleatorios (HTTP, HTTPS, Telnet, DNS, ICMP).

In [ ]:
import random
random.seed(42)
hosts = {1: ("00:00:00:00:00:01", "10.0.0.1"), 2: ("00:00:00:00:00:02", "10.0.0.2"),
         3: ("00:00:00:00:00:03", "10.0.1.3"), 4: ("00:00:00:00:00:04", "192.168.5.4")}
def traza(n=200):
    pkts = []
    for _ in range(n):
        a, b = random.sample(list(hosts), 2)
        proto, puerto = random.choice([(6, 80), (6, 443), (6, 23), (17, 53), (1, None)])
        p = {"in_port": a, "eth_src": hosts[a][0], "eth_dst": hosts[b][0],
             "ip_src": hosts[a][1], "ip_dst": hosts[b][1], "ip_proto": proto}
        if puerto: p["tp_dst"] = puerto
        pkts.append(p)
    return pkts
paquetes = traza()
print(paquetes[:3])

## 3. Controlador: switch con aprendizaje
La primera vez que ve una MAC de origen, el controlador aprende en qué puerto está. Si ya conoce el destino, **instala una regla** para que los siguientes paquetes no suban al controlador.

In [ ]:
class SwitchAprendiz:
    """Controlador tipo L2 learning switch: aprende MAC -> puerto e instala reglas."""
    def __init__(self):
        self.tabla = TablaFlujos(); self.mac_a_puerto = {}; self.packet_in = 0
    def recibir(self, p):
        acc = self.tabla.procesar(p)
        if acc == ["controller"]:
            self.packet_in += 1
            self.mac_a_puerto[p["eth_src"]] = p["in_port"]
            destino = self.mac_a_puerto.get(p["eth_dst"])
            if destino is None:
                return ["flood"]
            self.tabla.agregar(10, {"eth_dst": p["eth_dst"]}, [f"output:{destino}"])
            return [f"output:{destino}"]
        return acc

sw = SwitchAprendiz()
for p in paquetes:
    sw.recibir(p)
print("Packet-In recibidos por el controlador:", sw.packet_in)
sw.tabla.mostrar()

## 4. Ejercicios
**Ejercicio 1.** ¿Cuántos Packet-In recibió el controlador en la simulación anterior? Guárdelo en `resp_1` leyendo el atributo correspondiente, y explique por qué es mucho menor que 200.

In [ ]:
resp_1 = None

**Ejercicio 2. Firewall.** Cree una `TablaFlujos` que **bloquee Telnet** (TCP, puerto 23) y reenvíe todo lo demás por el puerto 1. Procese la traza y guarde en `resp_2` el número de paquetes descartados (use los contadores de la regla).

In [ ]:
fw = TablaFlujos()
# agregue sus reglas aquí

resp_2 = None

**Ejercicio 3.** Cree una nueva tabla con una regla de **mayor prioridad** que descarte todo el tráfico de `192.168.0.0/16` hacia `10.0.0.0/24`, más las reglas del ejercicio 2. Procese la traza y guarde en `resp_3` cuántos paquetes descartó esa nueva regla.

In [ ]:
resp_3 = None

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 61)
verificar('Ejercicio 2', resp_2, 39)
verificar('Ejercicio 3', resp_3, 28)

## 5. Del simulador a la realidad
En el laboratorio 1 repetirá esto con **Mininet** y un controlador **Ryu** en su computador:
```bash
sudo mn --topo single,4 --controller remote
ryu-manager ryu.app.simple_switch_13
```
Compare los Packet-In que observa con los de este simulador.